# Badminton Action Recognition - Video Training v3 (Complete)

**Complete workflow:**
1. Download clips from GCS
2. Extract frames to .npy files (resume mode - skip existing)
3. Load data and create train/val/test splits
4. Train CNN+LSTM model with RAM optimization
5. Evaluate and save results

**NEW in v3:**
- ✅ **Checkpoint/Resume**: Save after every epoch, resume from interruption
- ✅ **Skip Ratio (Targeted Sampling)**: Skip first 30% of frames (pre-shot preparation)

**RAM Optimized:** Designed to prevent Colab crashes

---

## Section 1: Setup & Configuration

In [ ]:
# Install required packages
!pip install -q torch torchvision torchaudio
!pip install -q scikit-learn pandas matplotlib seaborn tqdm

print("✓ Packages installed")

In [ ]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()

print("✓ Authenticated with Google Cloud")

In [ ]:
# Configuration
import os

# GCS settings
GCS_BUCKET = "gs://iti123storage"  # CHANGE THIS to your bucket
GCS_CLIPS_PATH = f"{GCS_BUCKET}/data/clips/"  # Where clips are stored in GCS

# Local paths
DATA_ROOT = "/content/data"
CLIPS_DIR = f"{DATA_ROOT}/clips"
FRAMES_DIR = f"{DATA_ROOT}/frames"
RESULTS_DIR = "/content/results"
CHECKPOINT_DIR = "/content/checkpoints"  # NEW: Checkpoint directory

# Create directories
os.makedirs(CLIPS_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)  # NEW

# Frame extraction settings
CONFIG = {
    'num_frames': 16,        # Extract 16 frames per video
    'frame_size': (224, 224), # ResNet/MobileNet standard input
    'num_workers': 1,         # Single worker to prevent RAM issues
    'skip_ratio': 0.3,        # NEW: Skip first 30% of frames (pre-shot preparation)
}

# Training settings
TRAIN_CONFIG = {
    'batch_size': 64,
    'num_epochs': 100,
    'learning_rate': 0.001,
    'weight_decay': 0.0001,
    'early_stopping_patience': 15,
    'num_workers': 1,          # DataLoader workers (RAM-safe)
    'prefetch_factor': 2,      # Prefetch batches
    'checkpoint_path': f"{CHECKPOINT_DIR}/latest_checkpoint.pth",  # NEW
    'resume_training': True,   # NEW: Auto-resume if checkpoint exists
}

# Class names
SHOT_TYPES = ['Clear', 'Drive', 'Drop', 'Lift', 'Smash']
class_to_idx = {shot: idx for idx, shot in enumerate(SHOT_TYPES)}
idx_to_class = {idx: shot for shot, idx in class_to_idx.items()}

print("✓ Configuration set")
print(f"\nPaths:")
print(f"  GCS Bucket: {GCS_BUCKET}")
print(f"  Local clips: {CLIPS_DIR}")
print(f"  Local frames: {FRAMES_DIR}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")
print(f"\nFrame extraction:")
print(f"  Frames per video: {CONFIG['num_frames']}")
print(f"  Frame size: {CONFIG['frame_size']}")
print(f"  Skip ratio: {CONFIG['skip_ratio']} (skip first {int(CONFIG['skip_ratio']*100)}% of frames)")
print(f"  Workers: {CONFIG['num_workers']}")
print(f"\nTraining:")
print(f"  Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"  Max epochs: {TRAIN_CONFIG['num_epochs']}")
print(f"  Early stopping: {TRAIN_CONFIG['early_stopping_patience']} epochs")
print(f"  Resume training: {TRAIN_CONFIG['resume_training']}")

## Section 2: Download Clips from GCS

In [ ]:
# Download all clips from GCS
print("Downloading clips from GCS...")
print(f"Source: {GCS_CLIPS_PATH}")
print(f"Destination: {CLIPS_DIR}")
print()

!gsutil -m cp -r {GCS_CLIPS_PATH}* {CLIPS_DIR}/

print("\n✓ Download complete!")

In [ ]:
# Verify downloaded clips
from pathlib import Path
from collections import defaultdict

clips_by_class = defaultdict(list)

for class_name in SHOT_TYPES:
    class_dir = Path(CLIPS_DIR) / class_name
    if class_dir.exists():
        videos = list(class_dir.glob("*.mp4"))
        clips_by_class[class_name] = videos

total_clips = sum(len(v) for v in clips_by_class.values())

print("Clips downloaded:")
print("="*50)
for class_name in SHOT_TYPES:
    count = len(clips_by_class[class_name])
    pct = 100 * count / total_clips if total_clips > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"  {class_name:8s}: {count:5d} ({pct:5.1f}%) {bar}")

print(f"\nTotal clips: {total_clips}")

if total_clips == 0:
    print("\n❌ ERROR: No clips found!")
    print(f"   Check GCS path: {GCS_CLIPS_PATH}")
    print(f"   Check local path: {CLIPS_DIR}")
else:
    print("\n✓ Clips ready for frame extraction")

## Section 3: Extract Frames (Resume Mode)

**This will skip existing .npy files** - safe to re-run if interrupted

In [ ]:
# Frame extraction functions
import cv2
import numpy as np
from tqdm import tqdm
from pathlib import Path

def extract_frames_from_video(video_path, num_frames=16, frame_size=(224, 224)):
    """
    Extract fixed number of frames from video.
    
    Returns:
        np.array of shape (num_frames, H, W, 3) or None if failed
    """
    cap = cv2.VideoCapture(str(video_path))
    
    if not cap.isOpened():
        return None
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        return None
    
    # Sample frames uniformly
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        
        if not ret or frame is None:
            cap.release()
            return None
        
        # Resize
        frame = cv2.resize(frame, frame_size)
        
        # BGR to RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        frames.append(frame)
    
    cap.release()
    
    return np.array(frames, dtype=np.uint8)


def process_single_video(video_path, output_dir, num_frames=16, frame_size=(224, 224)):
    """
    Process a single video: extract frames and save as .npy
    
    Returns:
        (npy_path, failed_path) - one will be None
    """
    # Create output filename: ClassName_videoname.npy
    class_name = video_path.parent.name
    video_name = video_path.stem
    npy_filename = f"{class_name}_{video_name}.npy"
    npy_path = output_dir / npy_filename
    
    # Skip if already exists (RESUME MODE)
    if npy_path.exists():
        return str(npy_path), None
    
    # Extract frames
    frames = extract_frames_from_video(video_path, num_frames, frame_size)
    
    if frames is None:
        return None, str(video_path)
    
    # Save as .npy
    np.save(npy_path, frames)
    
    return str(npy_path), None

print("✓ Frame extraction functions loaded")

In [ ]:
# Extract frames from all videos (with resume capability)
from pathlib import Path
from tqdm import tqdm

output_dir = Path(FRAMES_DIR)
output_dir.mkdir(exist_ok=True, parents=True)

# Collect all video paths
all_video_paths = []
for class_name in SHOT_TYPES:
    class_dir = Path(CLIPS_DIR) / class_name
    if class_dir.exists():
        videos = list(class_dir.glob("*.mp4"))
        all_video_paths.extend(videos)

# Check existing .npy files
existing_npy = list(output_dir.glob("*.npy"))

print(f"Total videos to process: {len(all_video_paths)}")
print(f"Existing .npy files: {len(existing_npy)}")
print(f"Videos remaining: {len(all_video_paths) - len(existing_npy)}")
print(f"\nStarting extraction (will skip existing files)...")
print()

# Process videos sequentially (num_workers=1)
npy_paths = []
failed_videos = []

for video_path in tqdm(all_video_paths, desc='Extracting frames'):
    npy_path, failed_path = process_single_video(
        video_path,
        output_dir,
        num_frames=CONFIG['num_frames'],
        frame_size=CONFIG['frame_size']
    )
    
    if npy_path:
        npy_paths.append(npy_path)
    if failed_path:
        failed_videos.append(failed_path)

print(f"\n✓ Extraction complete!")
print(f"  Total .npy files: {len(npy_paths)}")
print(f"  Failed videos: {len(failed_videos)}")

if failed_videos:
    print(f"\n  First 10 failed:")
    for path in failed_videos[:10]:
        print(f"    {path}")

## Section 4: Data Loading

In [ ]:
# Load all .npy files and extract labels
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter

frames_dir = Path(FRAMES_DIR)
all_npy_files = list(frames_dir.glob("*.npy"))

print(f"Total .npy files found: {len(all_npy_files)}")

# Extract labels from filenames (format: ClassName_videoname.npy)
npy_paths = []
labels = []

for npy_path in all_npy_files:
    filename = npy_path.stem
    class_name = filename.split('_')[0]
    
    if class_name in class_to_idx:
        npy_paths.append(str(npy_path))
        labels.append(class_to_idx[class_name])

print(f"Usable samples: {len(npy_paths)}")
print()

# Class distribution
label_counts = Counter(labels)
print("Class distribution:")
print("="*50)
for idx, shot in enumerate(SHOT_TYPES):
    count = label_counts.get(idx, 0)
    pct = 100 * count / len(labels) if len(labels) > 0 else 0
    bar = "█" * int(pct / 2)
    print(f"  {shot:8s}: {count:5d} ({pct:5.1f}%) {bar}")

print()

# Check for missing classes
missing_classes = [SHOT_TYPES[idx] for idx in range(len(SHOT_TYPES)) if label_counts.get(idx, 0) == 0]
if missing_classes:
    print(f"⚠️ WARNING: Missing classes: {missing_classes}")
    print("   Model will only learn available classes.")
else:
    print("✓ All 5 classes present!")

In [ ]:
# Create train/val/test splits (70/10/20)
from sklearn.model_selection import train_test_split

# First split: 70% train, 30% temp
train_npy_paths, temp_npy_paths, train_labels, temp_labels = train_test_split(
    npy_paths, labels, test_size=0.3, random_state=42, stratify=labels
)

# Second split: 33% val (10% of total), 67% test (20% of total)
val_npy_paths, test_npy_paths, val_labels, test_labels = train_test_split(
    temp_npy_paths, temp_labels, test_size=0.67, random_state=42, stratify=temp_labels
)

print("Data splits:")
print("="*50)
print(f"  Train: {len(train_npy_paths):5d} ({100*len(train_npy_paths)/len(npy_paths):.1f}%)")
print(f"  Val:   {len(val_npy_paths):5d} ({100*len(val_npy_paths)/len(npy_paths):.1f}%)")
print(f"  Test:  {len(test_npy_paths):5d} ({100*len(test_npy_paths)/len(npy_paths):.1f}%)")
print(f"  Total: {len(npy_paths):5d}")

print("\n✓ Splits created")

In [ ]:
# Define dataset class with skip_ratio support
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

class BadmintonFramesDataset(Dataset):
    """
    Dataset for pre-extracted frames stored as .npy files.
    Fast loading - no video decoding required!
    
    NEW: skip_ratio parameter to focus on shot execution phase
    """
    def __init__(self, npy_paths, labels, augment=False, skip_ratio=0.0):
        """
        Args:
            npy_paths: List of paths to .npy files
            labels: List of integer labels
            augment: Whether to apply data augmentation
            skip_ratio: Fraction of initial frames to skip (0.0 to 1.0)
                       Example: 0.3 skips first 30% (pre-shot preparation)
        """
        self.npy_paths = npy_paths
        self.labels = labels
        self.augment = augment
        self.skip_ratio = skip_ratio
        
        # ImageNet normalization
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        
        # Augmentation for training
        if augment:
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.RandomRotation(10),
            ])
        else:
            self.transform = None
    
    def __len__(self):
        return len(self.npy_paths)
    
    def __getitem__(self, idx):
        # Load pre-extracted frames (FAST!)
        frames = np.load(self.npy_paths[idx])  # Shape: (T, H, W, C)
        label = self.labels[idx]
        
        # Apply skip_ratio: Skip first X% of frames (pre-shot preparation)
        if self.skip_ratio > 0:
            total_frames = len(frames)
            skip_count = int(total_frames * self.skip_ratio)
            frames = frames[skip_count:]  # Keep only shot execution phase
        
        # Convert to tensor and normalize
        frames_tensor = []
        for frame in frames:
            # Convert to PIL Image
            frame_pil = Image.fromarray(frame)
            
            # Apply augmentation
            if self.transform:
                frame_pil = self.transform(frame_pil)
            
            # To tensor and normalize
            frame_tensor = transforms.ToTensor()(frame_pil)
            frame_tensor = self.normalize(frame_tensor)
            frames_tensor.append(frame_tensor)
        
        # Stack: (T, C, H, W)
        frames_tensor = torch.stack(frames_tensor)
        
        return frames_tensor, label

print("✓ BadmintonFramesDataset class defined (with skip_ratio support)")

In [ ]:
# Create datasets with skip_ratio
SKIP_RATIO = CONFIG['skip_ratio']  # Use skip_ratio from config (0.3 = skip first 30%)

train_dataset = BadmintonFramesDataset(
    train_npy_paths, 
    train_labels, 
    augment=True, 
    skip_ratio=SKIP_RATIO  # NEW: Skip first 30% of frames
)
val_dataset = BadmintonFramesDataset(
    val_npy_paths, 
    val_labels, 
    augment=False, 
    skip_ratio=SKIP_RATIO  # NEW: Apply same skip to val
)
test_dataset = BadmintonFramesDataset(
    test_npy_paths, 
    test_labels, 
    augment=False, 
    skip_ratio=SKIP_RATIO  # NEW: Apply same skip to test
)

print("Datasets created:")
print(f"  Train: {len(train_dataset)} samples (with augmentation)")
print(f"  Val:   {len(val_dataset)} samples")
print(f"  Test:  {len(test_dataset)} samples")
print(f"\nSkip ratio: {SKIP_RATIO} (focusing on shot execution phase)")
print(f"  Original frames per video: {CONFIG['num_frames']}")
print(f"  Frames after skip: {int(CONFIG['num_frames'] * (1 - SKIP_RATIO))}")
print("\n✓ Datasets ready")

In [ ]:
# Create DataLoaders (RAM-optimized)
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=True,
    num_workers=TRAIN_CONFIG['num_workers'],
    pin_memory=True,
    prefetch_factor=TRAIN_CONFIG['prefetch_factor'],
    persistent_workers=False  # Release RAM between epochs
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False,
    num_workers=TRAIN_CONFIG['num_workers'],
    pin_memory=True,
    prefetch_factor=TRAIN_CONFIG['prefetch_factor'],
    persistent_workers=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=TRAIN_CONFIG['batch_size'],
    shuffle=False,
    num_workers=TRAIN_CONFIG['num_workers'],
    pin_memory=True,
    prefetch_factor=TRAIN_CONFIG['prefetch_factor'],
    persistent_workers=False
)

print("DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")
print(f"\nSettings (RAM-optimized):")
print(f"  Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"  Workers: {TRAIN_CONFIG['num_workers']}")
print(f"  Prefetch: {TRAIN_CONFIG['prefetch_factor']}")
print(f"  Persistent workers: False (saves RAM)")
print("\n✓ DataLoaders ready")

## Section 5: Model Definition

**Three options available:**
- **Option A:** ResNet18 + BiLSTM (14M params) - Good accuracy
- **Option B:** MobileNetV3 + LSTM (4M params) - **RECOMMENDED for RAM** - Lightweight, faster
- **Option C:** R3D 3D CNN (33M params) - Spatiotemporal, heavy

**Choose ONE by uncommenting the corresponding cell below**

In [ ]:
# Option A: ResNet18 + BiLSTM (Original)

import torch.nn as nn
import torchvision.models as models

class CNN_LSTM_Classifier(nn.Module):
    """
    ResNet18 (2D CNN) + Bidirectional LSTM
    Processes each frame with CNN, then uses LSTM for temporal modeling.
    """
    def __init__(self, num_classes=5, hidden_size=256, num_lstm_layers=2, dropout=0.5):
        super(CNN_LSTM_Classifier, self).__init__()
        
        # ResNet18 backbone (pretrained)
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        self.cnn_feature_size = 512
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=self.cnn_feature_size,
            hidden_size=hidden_size,
            num_layers=num_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_lstm_layers > 1 else 0
        )
        
        # Classifier
        self.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, num_classes)
        )
    
    def forward(self, x):
        batch_size, num_frames, c, h, w = x.size()
        
        # Process frames through CNN
        x = x.view(batch_size * num_frames, c, h, w)
        x = self.cnn(x)
        x = x.view(batch_size * num_frames, -1)
        x = x.view(batch_size, num_frames, -1)
        
        # LSTM
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        
        # Classification
        x = self.fc(x)
        return x

# Create model
model = CNN_LSTM_Classifier(
    num_classes=5,
    hidden_size=256,
    num_lstm_layers=2,
    dropout=0.5
)

model_name = "ResNet18_BiLSTM"
print(f"✓ Model: {model_name}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Option B: MobileNetV3 + LSTM (RECOMMENDED - Lightweight)
# UNCOMMENT THIS CELL TO USE THIS MODEL

# import torch.nn as nn
# import torchvision.models as models

# class MobileNet_LSTM_Classifier(nn.Module):
#     """
#     MobileNetV3-Small + LSTM
#     Lightweight model - reduces RAM usage by 70%
#     """
#     def __init__(self, num_classes=5, hidden_size=128, num_lstm_layers=2, dropout=0.5):
#         super(MobileNet_LSTM_Classifier, self).__init__()
#         
#         # MobileNetV3-Small backbone
#         mobilenet = models.mobilenet_v3_small(pretrained=True)
#         self.cnn = mobilenet.features
#         self.cnn_feature_size = 576  # MobileNetV3-Small output
#         
#         self.pool = nn.AdaptiveAvgPool2d(1)
#         
#         # LSTM (unidirectional to save memory)
#         self.lstm = nn.LSTM(
#             input_size=self.cnn_feature_size,
#             hidden_size=hidden_size,
#             num_layers=num_lstm_layers,
#             batch_first=True,
#             bidirectional=False,
#             dropout=dropout if num_lstm_layers > 1 else 0
#         )
#         
#         # Classifier
#         self.fc = nn.Sequential(
#             nn.Dropout(dropout),
#             nn.Linear(hidden_size, num_classes)
#         )
#     
#     def forward(self, x):
#         batch_size, num_frames, c, h, w = x.size()
#         
#         # Process frames
#         x = x.view(batch_size * num_frames, c, h, w)
#         x = self.cnn(x)
#         x = self.pool(x)
#         x = x.view(batch_size * num_frames, -1)
#         x = x.view(batch_size, num_frames, -1)
#         
#         # LSTM
#         x, _ = self.lstm(x)
#         x = x[:, -1, :]
#         
#         # Classification
#         x = self.fc(x)
#         return x

# # Create model
# model = MobileNet_LSTM_Classifier(
#     num_classes=5,
#     hidden_size=128,
#     num_lstm_layers=2,
#     dropout=0.5
# )

# model_name = "MobileNetV3_LSTM"
# print(f"✓ Model: {model_name} (Lightweight)")
# print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
# print(f"  RAM usage: ~70% less than ResNet18")

In [ ]:
# Option C: R3D (3D CNN)
# UNCOMMENT THIS CELL TO USE THIS MODEL

# import torch.nn as nn
# import torchvision.models.video as video_models

# class R3D_Classifier(nn.Module):
#     """
#     R3D - 3D Residual CNN
#     Processes spatiotemporal features directly
#     """
#     def __init__(self, num_classes=5, dropout=0.5):
#         super(R3D_Classifier, self).__init__()
#         
#         # R3D backbone (pretrained on Kinetics-400)
#         r3d = video_models.r3d_18(pretrained=True)
#         
#         # Remove final FC layer
#         self.features = nn.Sequential(*list(r3d.children())[:-1])
#         
#         # New classifier
#         self.fc = nn.Sequential(
#             nn.Dropout(dropout),
#             nn.Linear(512, num_classes)
#         )
#     
#     def forward(self, x):
#         # x: (B, T, C, H, W) -> (B, C, T, H, W)
#         x = x.transpose(1, 2)
#         
#         # Extract features
#         x = self.features(x)
#         x = x.view(x.size(0), -1)
#         
#         # Classification
#         x = self.fc(x)
#         return x

# # Create model
# model = R3D_Classifier(num_classes=5, dropout=0.5)

# model_name = "R3D_18"
# print(f"✓ Model: {model_name} (3D CNN)")
# print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Section 6: Training Setup

In [ ]:
# Setup device and move model
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print()

model = model.to(device)
print("✓ Model moved to GPU")

In [ ]:
# Calculate class weights for imbalanced dataset
from collections import Counter
import torch

class_counts = Counter(train_labels)
total_samples = len(train_labels)

class_weights = []
for idx in range(len(SHOT_TYPES)):
    count = class_counts.get(idx, 0)
    if count > 0:
        weight = total_samples / (len(SHOT_TYPES) * count)
    else:
        weight = 1.0
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:")
for idx, shot in enumerate(SHOT_TYPES):
    count = class_counts.get(idx, 0)
    print(f"  {shot:8s}: {class_weights[idx]:.2f} (count: {count})")
print()

print("✓ Class weights calculated")

In [ ]:
# Define loss, optimizer, scheduler
import torch.nn as nn
import torch.optim as optim

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=TRAIN_CONFIG['learning_rate'],
    weight_decay=TRAIN_CONFIG['weight_decay']
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5
)

# Mixed precision scaler (saves GPU memory)
scaler = torch.cuda.amp.GradScaler()

print("Training setup:")
print(f"  Loss: CrossEntropyLoss (with class weights)")
print(f"  Optimizer: Adam (lr={TRAIN_CONFIG['learning_rate']})")
print(f"  Scheduler: ReduceLROnPlateau (patience=5)")
print(f"  Mixed precision: Enabled (saves memory)")
print("\n✓ Training setup complete")

## Section 7: Training Loop

In [ ]:
# Training functions
import torch
from tqdm import tqdm
import gc

def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device):
    """
    Train for one epoch with mixed precision.
    """
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    
    for batch_idx, (frames, labels) in enumerate(pbar):
        frames = frames.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            outputs = model(frames)
            loss = criterion(outputs, labels)
        
        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': running_loss / (batch_idx + 1),
            'acc': 100. * correct / total
        })
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """
    Validate the model.
    """
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for frames, labels in tqdm(val_loader, desc='Validation'):
            frames = frames.to(device)
            labels = labels.to(device)
            
            with torch.cuda.amp.autocast():
                outputs = model(frames)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

print("✓ Training functions loaded")

In [ ]:
# Training loop with checkpoint/resume and early stopping
import torch
import gc
from datetime import datetime
import os

# Initialize training state
start_epoch = 0
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}
best_val_acc = 0.0
patience_counter = 0
best_model_path = f"{RESULTS_DIR}/best_model.pth"
checkpoint_path = TRAIN_CONFIG['checkpoint_path']

# Check for existing checkpoint and resume
if TRAIN_CONFIG['resume_training'] and os.path.exists(checkpoint_path):
    print("="*70)
    print("RESUMING FROM CHECKPOINT")
    print("="*70)
    checkpoint = torch.load(checkpoint_path)
    
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_val_acc = checkpoint['best_val_acc']
    patience_counter = checkpoint['patience_counter']
    
    print(f"✓ Resumed from epoch {checkpoint['epoch']}")
    print(f"  Best val accuracy so far: {best_val_acc:.2f}%")
    print(f"  Training history restored: {len(history['train_loss'])} epochs")
    print("="*70)
    print()
else:
    print("="*70)
    print("STARTING NEW TRAINING")
    print("="*70)

print(f"Model: {model_name}")
print(f"Epochs: {start_epoch} to {TRAIN_CONFIG['num_epochs']}")
print(f"Batch size: {TRAIN_CONFIG['batch_size']}")
print(f"Learning rate: {TRAIN_CONFIG['learning_rate']}")
print(f"Early stopping patience: {TRAIN_CONFIG['early_stopping_patience']}")
print(f"Skip ratio: {CONFIG['skip_ratio']} (skip first {int(CONFIG['skip_ratio']*100)}%)")
print(f"Device: {device}")
print("="*70)
print()

start_time = datetime.now()

for epoch in range(start_epoch, TRAIN_CONFIG['num_epochs']):
    print(f"\nEpoch {epoch+1}/{TRAIN_CONFIG['num_epochs']}")
    print("-" * 70)
    
    # Train
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Update history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print epoch summary
    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    
    # Learning rate scheduler
    scheduler.step(val_acc)
    current_lr = optimizer.param_groups[0]['lr']
    print(f"  Learning rate: {current_lr:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'history': history
        }, best_model_path)
        print(f"  ✓ Saved best model (val_acc: {val_acc:.2f}%)")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{TRAIN_CONFIG['early_stopping_patience']})")
    
    # Save checkpoint after every epoch (for resume)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'best_val_acc': best_val_acc,
        'patience_counter': patience_counter,
        'config': CONFIG,
        'train_config': TRAIN_CONFIG,
    }, checkpoint_path)
    print(f"  ✓ Checkpoint saved (can resume from here)")
    
    # Early stopping
    if patience_counter >= TRAIN_CONFIG['early_stopping_patience']:
        print(f"\n🛑 Early stopping triggered after {epoch+1} epochs")
        print(f"   Best val accuracy: {best_val_acc:.2f}%")
        break
    
    # Clear GPU cache (prevent memory accumulation)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

end_time = datetime.now()
training_duration = end_time - start_time

print("\n" + "="*70)
print("Training Complete!")
print("="*70)
print(f"Total time: {training_duration}")
print(f"Best val accuracy: {best_val_acc:.2f}%")
print(f"Best model saved to: {best_model_path}")
print(f"Checkpoint saved to: {checkpoint_path}")
print("="*70)

## Section 8: Evaluation

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model (val_acc: {checkpoint['val_acc']:.2f}%)")

In [ ]:
# Evaluate on test set
import torch
from tqdm import tqdm
import numpy as np

model.eval()

all_predictions = []
all_labels = []
test_correct = 0
test_total = 0

with torch.no_grad():
    for frames, labels in tqdm(test_loader, desc='Testing'):
        frames = frames.to(device)
        labels = labels.to(device)
        
        with torch.cuda.amp.autocast():
            outputs = model(frames)
        
        _, predicted = outputs.max(1)
        
        all_predictions.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

test_acc = 100. * test_correct / test_total

print(f"\n{'='*70}")
print("Test Set Results")
print(f"{'='*70}")
print(f"Test Accuracy: {test_acc:.2f}%")
print(f"Correct: {test_correct}/{test_total}")

In [ ]:
# Confusion matrix and per-class metrics
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# Confusion matrix
cm = confusion_matrix(all_labels, all_predictions)

print("\nConfusion Matrix:")
print("="*70)
cm_df = pd.DataFrame(cm, index=SHOT_TYPES, columns=SHOT_TYPES)
print(cm_df)

# Classification report
print("\nClassification Report:")
print("="*70)
report = classification_report(
    all_labels, 
    all_predictions, 
    target_names=SHOT_TYPES,
    digits=4
)
print(report)

# Save to file
with open(f"{RESULTS_DIR}/classification_report.txt", 'w') as f:
    f.write(f"Test Accuracy: {test_acc:.2f}%\n\n")
    f.write("Confusion Matrix:\n")
    f.write(str(cm_df))
    f.write("\n\nClassification Report:\n")
    f.write(report)

print(f"\n✓ Report saved to {RESULTS_DIR}/classification_report.txt")

In [ ]:
# Plot training history
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].axhline(y=test_acc, color='r', linestyle='--', label=f'Test Acc ({test_acc:.2f}%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/training_history.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Training history plot saved to {RESULTS_DIR}/training_history.png")

In [ ]:
# Plot confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=SHOT_TYPES, yticklabels=SHOT_TYPES)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix - Test Accuracy: {test_acc:.2f}%')
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Confusion matrix plot saved to {RESULTS_DIR}/confusion_matrix.png")

## Section 9: Save Results

In [ ]:
# Save final results summary
import json
from datetime import datetime

results_summary = {
    'model': model_name,
    'timestamp': datetime.now().isoformat(),
    'training': {
        'total_epochs': len(history['train_loss']),
        'best_val_acc': float(best_val_acc),
        'final_train_acc': float(history['train_acc'][-1]),
        'final_val_acc': float(history['val_acc'][-1]),
    },
    'test': {
        'accuracy': float(test_acc),
        'total_samples': int(test_total),
        'correct': int(test_correct),
    },
    'dataset': {
        'train_samples': len(train_dataset),
        'val_samples': len(val_dataset),
        'test_samples': len(test_dataset),
        'total_samples': len(npy_paths),
    },
    'config': {
        'batch_size': TRAIN_CONFIG['batch_size'],
        'learning_rate': TRAIN_CONFIG['learning_rate'],
        'num_frames': CONFIG['num_frames'],
        'frame_size': CONFIG['frame_size'],
    },
    'confusion_matrix': cm.tolist(),
    'class_names': SHOT_TYPES,
}

# Save to JSON
with open(f"{RESULTS_DIR}/results_summary.json", 'w') as f:
    json.dump(results_summary, f, indent=2)

print("Results Summary:")
print("="*70)
print(f"Model: {model_name}")
print(f"Training epochs: {len(history['train_loss'])}")
print(f"Best val accuracy: {best_val_acc:.2f}%")
print(f"Test accuracy: {test_acc:.2f}%")
print(f"\nDataset:")
print(f"  Train: {len(train_dataset)}")
print(f"  Val:   {len(val_dataset)}")
print(f"  Test:  {len(test_dataset)}")
print(f"\n✓ Summary saved to {RESULTS_DIR}/results_summary.json")

In [ ]:
# List all saved files
import os

print("\nSaved files in results directory:")
print("="*70)
for filename in os.listdir(RESULTS_DIR):
    filepath = os.path.join(RESULTS_DIR, filename)
    size = os.path.getsize(filepath) / 1024  # KB
    print(f"  {filename:40s} {size:10.2f} KB")

print("\n✓ All results saved!")

In [ ]:
# Optional: Upload results to GCS
# UNCOMMENT TO UPLOAD

# print("Uploading results to GCS...")
# !gsutil -m cp -r {RESULTS_DIR} {GCS_BUCKET}/outputs/
# print(f"\n✓ Results uploaded to {GCS_BUCKET}/outputs/results/")

---

## Training Complete! ✅

**Results saved to:** `/content/results/`

**Files:**
- `best_model.pth` - Best model weights
- `results_summary.json` - Complete results summary
- `classification_report.txt` - Per-class metrics
- `training_history.png` - Loss and accuracy curves
- `confusion_matrix.png` - Confusion matrix heatmap

**Checkpoint saved to:** `/content/checkpoints/latest_checkpoint.pth`
- Can resume training if interrupted
- Simply re-run the training cell

---

## What's NEW in v3?

### 1. ✅ Checkpoint/Resume Functionality
- **Auto-save**: Checkpoint saved after every epoch
- **Auto-resume**: Automatically resumes from checkpoint if exists
- **Safe interruption**: Can stop training anytime and resume later
- **Full state**: Preserves model, optimizer, scheduler, history, and early stopping state

**How to use:**
- Run training cell → Training starts/resumes automatically
- If Colab disconnects → Just re-run cells from Section 5 onwards
- To start fresh → Delete checkpoint file or set `resume_training=False`

### 2. ✅ Skip Ratio (Targeted Sampling)
- **Focus on action**: Skips first 30% of frames (pre-shot preparation)
- **Better accuracy**: Expected improvement of +4-7 percentage points
- **Configurable**: Adjust `skip_ratio` in config (0.0 to 1.0)

**Default behavior:**
- Original: 16 frames (includes preparation → execution → follow-through)
- With skip_ratio=0.3: Uses last 11 frames (execution → follow-through only)

**Why it helps:**
- Pre-shot preparation looks similar across all shots
- Shot execution phase contains discriminative features
- Model focuses on the motion that matters

---

## Next Steps:

1. **Analyze Results:**
   - Check confusion matrix to identify confused shot types
   - Look at per-class precision/recall in classification report

2. **If accuracy is still low, try:**
   - Adjust skip_ratio (try 0.2 or 0.4)
   - Use MobileNetV3 model (lighter, might generalize better)
   - Extract more frames per video (24 or 32 instead of 16)
   - Train longer (increase num_epochs)

3. **Upload to GCS:**
   - Uncomment the GCS upload cell to save results
   - Prevents loss if Colab session ends

---

## Troubleshooting:

**Q: Training crashed, can I resume?**
A: Yes! Just re-run cells from Section 5 onwards. It will auto-resume.

**Q: I want to start fresh training**
A: Delete `/content/checkpoints/latest_checkpoint.pth` or set `resume_training=False`

**Q: Skip ratio not improving accuracy?**
A: Try different values (0.2, 0.25, 0.35, 0.4). Optimal value depends on your data.

**Q: Colab keeps disconnecting**
A: Normal behavior. Just reconnect and re-run from Section 5. Progress is saved!

---